# Consumption Model Probe — Phase 2 Go/No-Go Gate

**Purpose:** Validate that *your* TeslaMate drive history supports the 4-variable consumption model **before** you commit to `tesla nav consumption calibrate` as the source of truth for trip planning.

**Decision table (from `.omc/plans/native-ev-planner.md` Phase 2 Step 0):**

| Metric | Action |
|---|---|
| mean R² ≥ 0.7 AND MAPE ≤ 10% | **PROCEED** — ship calibrated model |
| 0.5 ≤ mean R² < 0.7 | **FALLBACK** — single-variable model (base + speed) |
| mean R² < 0.5 | **INVALIDATED** — stay on Phase 1 + ABRP deep link |

**Optional dependencies (install on top of tesla-cli):**
```
pip install scikit-learn pandas matplotlib
```
If these aren't installed, the notebook still runs — it just falls back to the built-in fit.


In [ ]:
# 1. Connect to TeslaMate via tesla_cli config
from tesla_cli.core.config import load_config
from tesla_cli.core.backends.teslaMate import TeslaMateBacked

cfg = load_config()
assert cfg.teslaMate.database_url, (
    'TeslaMate not configured. Run: tesla teslaMate connect <postgres_url>'
)
tm = TeslaMateBacked(cfg.teslaMate.database_url, car_id=cfg.teslaMate.car_id)
print('connected to TeslaMate at', cfg.teslaMate.database_url[:40], '...')

In [ ]:
# 2. Pull the calibration dataset (last 90 days of drives)
segments = tm.get_calibration_dataset(days=90, battery_kwh=75.0)
print(f'segments: {len(segments)}')
segments[:3]

In [ ]:
# 3. Feature engineering table (requires pandas)
try:
    import pandas as pd
    df = pd.DataFrame(segments)
    print(df.describe())
except ImportError:
    print('pandas not installed — skipping feature table. Install: pip install pandas')

In [ ]:
# 4. 5-fold cross-validation using the built-in fit.
#    scikit-learn is optional; we implement a simple K-fold split by hand.
import random
import statistics
from tesla_cli.core.planner.consumption import fit_from_dataset, estimate_wh_per_km

CAR_MODEL = cfg.planner.default_car_model or 'tesla:my:22:bt37:lr'
random.seed(42)
segs = list(segments)
random.shuffle(segs)
K = 5
fold_size = len(segs) // K

def score(model, holdout):
    ys = [s['energy_kwh'] * 1000.0 for s in holdout]
    preds = [
        estimate_wh_per_km(
            model, s['avg_speed_kmh'], s['elevation_delta_m'],
            s.get('outside_temp_c'), s['distance_km']) * s['distance_km']
        for s in holdout
    ]
    mape = 100.0 * statistics.mean(abs(y - p) / max(y, 1.0) for y, p in zip(ys, preds))
    y_mean = statistics.mean(ys)
    ss_tot = sum((y - y_mean) ** 2 for y in ys)
    ss_res = sum((y - p) ** 2 for y, p in zip(ys, preds))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return r2, mape

r2s, mapes = [], []
for k in range(K):
    holdout = segs[k * fold_size : (k + 1) * fold_size]
    train = segs[: k * fold_size] + segs[(k + 1) * fold_size :]
    if len(train) < 30:
        print(f'fold {k}: not enough training samples ({len(train)}); skipping')
        continue
    m = fit_from_dataset(CAR_MODEL, train)
    r2, mape = score(m, holdout)
    r2s.append(r2); mapes.append(mape)
    print(f'fold {k}:  R²={r2:.3f}  MAPE={mape:.1f}%')

if r2s:
    mean_r2 = statistics.mean(r2s)
    mean_mape = statistics.mean(mapes)
    ci = 1.96 * statistics.pstdev(r2s) / (len(r2s) ** 0.5) if len(r2s) > 1 else 0.0
    print(f'\nmean R² = {mean_r2:.3f} ± {ci:.3f} (95% CI)')
    print(f'mean MAPE = {mean_mape:.1f}%')
else:
    print('insufficient data for 5-fold CV — use the built-in fit without CV.')

In [ ]:
# 5. Decision table printout
if r2s:
    if mean_r2 >= 0.7 and mean_mape <= 10.0:
        print('DECISION: PROCEED with Phase 2 calibrated model.')
    elif mean_r2 >= 0.5:
        print('DECISION: FALLBACK — single-variable (base + speed) model only.')
    else:
        print('DECISION: INVALIDATED — stay on Phase 1 + ABRP deep link.')

In [ ]:
# 6. Final cell: save the fitted model to ~/.tesla-cli/consumption.toml.
#    Only run this if DECISION == PROCEED (or FALLBACK and you accept the reduced accuracy).
from tesla_cli.core.planner.consumption import save_calibrated
final_model = fit_from_dataset(CAR_MODEL, segments)
save_calibrated(final_model)
print('saved to ~/.tesla-cli/consumption.toml')
print(final_model.model_dump_json(indent=2))